# Train DepthBrain — Vidgin's own depth estimator (one session)

DepthBrain (`vidgen/depthbrain.py`, ~9.3M params) starts from **random
weights** and learns from the MiDaS depth maps (`*_depth.npy`) already in
your Round 1–4 datasets. MiDaS is only the teacher that made those labels —
it is not downloaded, loaded or shipped by this notebook.

What this notebook does, in order:

1. Mount Google Drive, check the GPU.
2. Clone the repo, install `requirements.txt`.
3. **Stage** the Round 1–4 data from Drive to local disk
   (`scripts/stage_depth_data.py`) — the trainer must not read from the
   Drive mount.
4. **Dry run** (`train_depthbrain.py --dry-run`): clips == depth maps ==
   manifest lines, and at least 50 clips held out. No training.
5. **Train, first session** — stops after `FIRST_SESSION_STEPS` with a
   checkpoint on Drive (every 500 steps).
6. **Resume** from the newest checkpoint on Drive and train to `STEPS`.
   If Colab disconnects at any point, re-run cells 1–4 and then this cell:
   it continues from the last checkpoint.
7. **Report**: steps, final loss, every gate metric (incl. Pearson), PASS/FAIL.

Every step is the repo's own command line (the same commands work on any
machine); the notebook only runs them. A PASS does **not** change any
default: prep keeps `--depth-source midas` until the owner decides.

In [ ]:
# --- 0. Settings: edit these, then Run all ---
# Your Round 1-4 dataset folders on Drive (each: manifest.jsonl, clip .npy
# files and their *_depth.npy MiDaS maps).
DRIVE_DATA = [
    "/content/drive/MyDrive/vidgin-data/round1",
    "/content/drive/MyDrive/vidgin-data/round2",
    "/content/drive/MyDrive/vidgin-data/round3",
    "/content/drive/MyDrive/vidgin-data/round4",
]
DRIVE_CKPT = "/content/drive/MyDrive/vidgin-checkpoints/depthbrain"  # checkpoints + report (survive the session)
LOCAL_DATA = "/content/depth_data"     # local disk copy the trainer reads
REPO_URL = "https://github.com/shahrozkalash/Vidgin.git"
REPO_REF = "main"                      # branch or commit to train from

STEPS = 20000             # total steps (across sessions)
FIRST_SESSION_STEPS = 500 # session 1 stops here, then cell 6 resumes (proves resume)
SAVE_EVERY = 500          # checkpoint to Drive every N steps
VAL_PCT = 5.0             # % of clips held out (never trained on); raise it if the dry run says < 50
BATCH_SIZE = 8
AMP = "fp16"              # T4 / P100: fp16; A100 / L4: bf16 also works

In [ ]:
# --- 1. Mount Drive and check the GPU ---
import os, sys, json, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("not on Colab: no Drive mount (set DRIVE_DATA to local / Kaggle input folders)")
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (CPU is far too slow for this)")
for d in DRIVE_DATA:
    print(("ok      " if os.path.isfile(os.path.join(d, "manifest.jsonl")) else "MISSING ") + d)

In [ ]:
# --- 2. Get the code and install requirements ---
REPO = "/content/Vidgin" if os.path.isdir("/content") else os.path.abspath("Vidgin")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "origin"], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "--detach",
                REPO_REF if len(REPO_REF) == 40 else f"origin/{REPO_REF}"], check=True)
os.chdir(REPO)
HEAD = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("training from commit", HEAD)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)


def run(args):
    """Run one repo command, stream its output, stop the notebook if it fails."""
    cmd = [sys.executable] + [str(a) for a in args]
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise SystemExit(f"command failed (exit {p.returncode}) - see the output above")

In [ ]:
# --- 3. Stage the data from Drive to local disk ---
# Copies manifest.jsonl + every clip .npy + its *_depth.npy, nothing else.
# Re-running skips files that are already staged.
run(["scripts/stage_depth_data.py", *DRIVE_DATA, "--dst", LOCAL_DATA])
DATA_DIRS = json.load(open(os.path.join(LOCAL_DATA, "staged.json")))["dirs"]
print("trainer will read:", DATA_DIRS)

In [ ]:
# --- 4. Dry run: check the staged data, no training ---
# Fails (and stops here) unless clips == depth maps == manifest lines in
# every folder and at least 50 clips are held out for the agreement check.
run(["scripts/train_depthbrain.py", "--data", *DATA_DIRS, "--val-pct", VAL_PCT, "--dry-run"])

In [ ]:
# --- 5. Train: first session (stops at FIRST_SESSION_STEPS) ---
# --resume auto: continue DRIVE_CKPT/latest.pt if it exists, else start fresh.
# A checkpoint is written to Drive every SAVE_EVERY steps.
TRAIN = ["scripts/train_depthbrain.py", "--data", *DATA_DIRS,
         "--resume", "auto", "--ckpt-dir", DRIVE_CKPT,
         "--steps", STEPS, "--save-every", SAVE_EVERY,
         "--batch-size", BATCH_SIZE, "--amp", AMP, "--val-pct", VAL_PCT,
         "--log", os.path.join(DRIVE_CKPT, "train.jsonl")]
run(TRAIN + ["--session-steps", FIRST_SESSION_STEPS])

In [ ]:
# --- 6. Resume from the newest checkpoint and train to STEPS ---
# The same command as cell 5: it finds DRIVE_CKPT/latest.pt and prints
# "resumed at step N". After a disconnect, re-run cells 0-4 and then this one.
assert os.path.isfile(os.path.join(DRIVE_CKPT, "latest.pt")), "no checkpoint on Drive yet - run cell 5"
run(TRAIN)

In [ ]:
# --- 7. Report: steps, final loss, gate metrics (incl. Pearson), PASS/FAIL ---
run(["scripts/train_depthbrain.py", "--summary", os.path.join(DRIVE_CKPT, "depthbrain_report.json")])
print("\nfiles on Drive:", sorted(os.listdir(DRIVE_CKPT)))

## After the run

- `depthbrain.pt` and `depthbrain_report.json` are in `DRIVE_CKPT`.
- **PASS** means DepthBrain agrees with MiDaS on clips it never trained on,
  by every threshold of the gate. It is agreement with the teacher, not
  true depth. Prep still defaults to MiDaS; switching is the owner's call.
- **FAIL** lists every threshold missed. Training longer: raise `STEPS` and
  re-run cell 6 (it resumes; the learning-rate schedule stays at its floor
  past the original `STEPS`).